
# Comparaison Google Books / OpenLibrary

Objectif :

 - comparer la couverture des deux APIs ;
 - comparer la complétude des métadonnées ;
 - identifier les ISBN trouvés ou non selon les sources ;
 - déterminer si une stratégie multi-sources est nécessaire.

In [1]:

from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]

## Chargement des résultats

In [2]:

google_path = PROJECT_ROOT / "data" / "processed" / "google_books" / "google_books_results_full.csv"
openlibrary_path = PROJECT_ROOT / "data" / "processed" / "openlibrary" / "openlibrary_results_full.csv"

df_google = pd.read_csv(google_path)
df_openlibrary = pd.read_csv(openlibrary_path)

df_google["source"] = "google_books"
df_openlibrary["source"] = "openlibrary"

df_results = pd.concat(
    [df_google, df_openlibrary],
    ignore_index=True,
)

df_results.head()

,source,isbn_query,found,error,source_id,google_books_id,isbn,title,subtitle,authors,...,average_rating,ratings_count,preview_link,info_link,canonical_volume_link,industry_identifiers,raw_data,status_code,openlibrary_key,publishers
0,google_books,9782351423554,True,NaN,f4RwPgAACAAJ,f4RwPgAACAAJ,9.782351e+12,Vinland Saga,NaN,"['Makoto Yukimura', 'Xavière Daumarie']",...,NaN,NaN,http://books.google.fr/books?id=f4RwPgAACAAJ&d...,http://books.google.fr/books?id=f4RwPgAACAAJ&d...,https://books.google.com/books/about/Vinland_S...,"[{'type': 'ISBN_10', 'identifier': '2351423550...","{'kind': 'books#volume', 'id': 'f4RwPgAACAAJ',...",200,NaN,NaN
1,google_books,9791042019396,False,no_result,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,200,NaN,NaN
2,google_books,9782820354488,False,no_result,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,200,NaN,NaN
3,google_books,9782290042298,True,NaN,U4qHPQAACAAJ,U4qHPQAACAAJ,9.782290e+12,Le précepteur du héros,NaN,['Riku Sanjô'],...,NaN,NaN,http://books.google.fr/books?id=U4qHPQAACAAJ&d...,http://books.google.fr/books?id=U4qHPQAACAAJ&d...,https://books.google.com/books/about/Le_pr%C3%...,"[{'type': 'ISBN_10', 'identifier': '2290042293...","{'kind': 'books#volume', 'id': 'U4qHPQAACAAJ',...",200,NaN,NaN
4,google_books,9782845808331,True,NaN,xaBwPAAACAAJ,xaBwPAAACAAJ,9.782846e+12,Dragon quest : la quête de Daï,NaN,"['Riku Sanjô', 'Kōji Inada, 1964-']",...,NaN,NaN,http://books.google.fr/books?id=xaBwPAAACAAJ&d...,http://books.google.fr/books?id=xaBwPAAACAAJ&d...,https://books.google.com/books/about/Dragon_qu...,"[{'type': 'ISBN_10', 'identifier': '284580833X...","{'kind': 'books#volume', 'id': 'xaBwPAAACAAJ',...",200,NaN,NaN


## Taux de couverture par source

In [3]:

coverage_by_source = (
    df_results
    .groupby("source")["found"]
    .mean()
    .mul(100)
    .round(2)
    .reset_index(name="coverage_rate")
)

coverage_by_source

,source,coverage_rate
0,google_books,50.0
1,openlibrary,20.0


## Couverture par isbn

In [4]:

df_coverage_by_isbn = (
    df_results
    .pivot_table(
        index="isbn_query",
        columns="source",
        values="found",
        aggfunc="first",
    )
    .reset_index()
)

for source in ["google_books", "openlibrary"]:
    if source not in df_coverage_by_isbn.columns:
        df_coverage_by_isbn[source] = False

df_coverage_by_isbn["sources_found_count"] = (
    df_coverage_by_isbn[["google_books", "openlibrary"]]
    .fillna(False)
    .astype(bool)
    .sum(axis=1)
)

df_coverage_by_isbn["found_anywhere"] = (
    df_coverage_by_isbn["sources_found_count"] > 0
)

df_coverage_by_isbn

source,isbn_query,google_books,openlibrary,sources_found_count,found_anywhere
0,9782203001190,True,True,2,True
1,9782290042298,True,True,2,True
2,9782351423554,True,True,2,True
3,9782374123035,False,False,0,False
4,9782374126173,False,True,1,True
5,9782384967421,False,False,0,False
6,9782413042341,True,False,1,True
7,9782820354488,False,False,0,False
8,9782822244787,True,False,1,True
9,9782845808331,True,False,1,True


## Couverture cumulée

In [6]:
global_coverage = round(
    df_coverage_by_isbn["found_anywhere"].mean() * 100,
    2,
)

print(
    f"Couverture cumulée Google Books + OpenLibrary : {global_coverage:.2f}%"
)

Couverture cumulée Google Books + OpenLibrary : 55.00%


## ISBN trouvé par les deux sources

In [7]:

df_found_both = df_coverage_by_isbn[
    (df_coverage_by_isbn["google_books"] == True)
    & (df_coverage_by_isbn["openlibrary"] == True)
]

df_found_both

source,isbn_query,google_books,openlibrary,sources_found_count,found_anywhere
0,9782203001190,True,True,2,True
1,9782290042298,True,True,2,True
2,9782351423554,True,True,2,True


## ISBN trouvé uniquement par google books

In [8]:

df_only_google = df_coverage_by_isbn[
    (df_coverage_by_isbn["google_books"] == True)
    & (df_coverage_by_isbn["openlibrary"] != True)
]

df_only_google

source,isbn_query,google_books,openlibrary,sources_found_count,found_anywhere
6,9782413042341,True,False,1,True
8,9782822244787,True,False,1,True
9,9782845808331,True,False,1,True
10,9782858150045,True,False,1,True
11,9791026854920,True,False,1,True
13,9791038206229,True,False,1,True
16,9791039143431,True,False,1,True


## ISBN trouvé uniquement par open library

In [9]:

df_only_openlibrary = df_coverage_by_isbn[
    (df_coverage_by_isbn["google_books"] != True)
    & (df_coverage_by_isbn["openlibrary"] == True)
]

df_only_openlibrary

source,isbn_query,google_books,openlibrary,sources_found_count,found_anywhere
4,9782374126173,False,True,1,True


## Completude des metadonnées par source

In [10]:

metadata_fields = [
    "title",
    "authors",
    "publisher",
    "published_date",
    "language",
    "description",
    "categories",
    "thumbnail",
]

available_fields = [
    field for field in metadata_fields
    if field in df_results.columns
]

metadata_completeness = (
    df_results
    .groupby("source")[available_fields]
    .apply(lambda df: df.notna().mean())
    .mul(100)
    .round(2)
)

metadata_completeness

,title,authors,publisher,published_date,language,description,categories,thumbnail
source,,,,,,,,
google_books,50.0,50.0,10.0,50.0,50.0,20.0,50.0,5.0
openlibrary,20.0,20.0,20.0,20.0,0.0,0.0,20.0,15.0


## Analyse des erreurs par sources

In [11]:

error_summary = (
    df_results[df_results["found"] == False]
    .groupby(["source", "error"])
    .size()
    .reset_index(name="count")
)

error_summary

,source,error,count
0,google_books,no_result,10
1,openlibrary,no_result,16


## Export des résultats comparatifs

In [12]:
output_dir = PROJECT_ROOT / "data" / "processed" / "api_comparison"
output_dir.mkdir(parents=True, exist_ok=True)

coverage_by_source.to_csv(output_dir / "coverage_by_source.csv", index=False)
df_coverage_by_isbn.to_csv(output_dir / "coverage_by_isbn.csv", index=False)
metadata_completeness.to_csv(output_dir / "metadata_completeness_by_source.csv")
error_summary.to_csv(output_dir / "error_summary_by_source.csv", index=False)

output_dir

WindowsPath('c:/Users/yoann/Documents/mes projets/CollectionLens V2/data/processed/api_comparison')

# Premiers constats

La comparaison entre Google Books et OpenLibrary montre une couverture cumulée de 55 % sur l’échantillon de 20 ISBN testés.

## Couverture des sources

Google Books obtient une meilleure couverture globale que OpenLibrary sur ce mini-dataset.

OpenLibrary apporte quelques résultats complémentaires, mais la couverture cumulée reste insuffisante pour couvrir correctement le périmètre mangas, BD et comics.

## Complétude des métadonnées

Google Books présente une meilleure complétude sur plusieurs champs structurants :

- titre : 50 % ;
- auteurs : 50 % ;
- date de publication : 50 % ;
- langue : 50 % ;
- catégories : 50 %.

En revanche, certains champs restent très faibles :

- éditeur : 10 % ;
- description : 20 % ;
- miniature : 5 %.

OpenLibrary présente une complétude plus faible sur l’ensemble des champs, avec notamment :

- titre : 20 % ;
- auteurs : 20 % ;
- éditeur : 20 % ;
- date de publication : 20 % ;
- description : 0 % ;
- langue : 0 %.

## Limites identifiées

Les deux APIs généralistes montrent des limites importantes :

- couverture insuffisante des ISBN testés ;
- faible disponibilité des descriptions ;
- métadonnées hétérogènes selon les sources ;
- faible couverture des éditions françaises récentes ;
- nécessité de gérer les absences de résultats et les erreurs API.

## Conclusion POC

Google Books et OpenLibrary sont utiles comme premières sources bibliographiques généralistes, mais ne suffisent pas comme sources uniques pour CollectionLens.

Ces résultats confirment la nécessité de poursuivre le POC vers :

- une stratégie multi-sources ;
- l’évaluation de sources complémentaires plus spécialisées ;
- la mise en place d’un cache local ;
- la normalisation des métadonnées ;
- la documentation des limites de chaque source.